<a href="https://colab.research.google.com/github/hibbo67/ai-engineering-101/blob/main/day23_llm_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb google-genai
from google.colab import userdata
from google import genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

# Load key gracefully
api_keys = []
for i in range(1, 7):
    try:
        key = userdata.get(f"GEMINI_KEY_{i}")
        if key:
            api_keys.append(key)
    except Exception:
        pass
if not api_keys:
    raise ValueError("No GEMINI_KEYs found in Colab userdata.")

embed_client = genai.Client(api_key=api_keys[0])

# 3. Embedding function
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self):
        pass # Silences the Chroma deprecation warning

    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            result = embed_client.models.embed_content(
                model="gemini-embedding-2",
                contents=[text],
            )
            vectors.append([float(x) for x in result.embeddings[0].values])
        return vectors

print(f"Setup complete. {len(ALL_KEYS)} keys loaded for manual rotation.")

Setup complete. 3 keys loaded for manual rotation.


In [ ]:
docs = [
    "Our refund policy allows returns within 30 days of purchase.",
    "Shipping is free for orders over $50.",
    "We ship to the US, Canada, and EU only.",
    "Customer support is available 9am–5pm EST, Monday to Friday.",
    "Digital products cannot be refunded once downloaded.",
    "To initiate a return, please contact support with your order number.",
]

In [ ]:
# Initialize the ChromaDB client (in-memory persistent client or ephemeral client)
db = chromadb.Client()

In [ ]:
# Get the collection (creates it if it doesn't exist, avoids crash if it does)
col = db.get_or_create_collection(
    name="support_kb",
    embedding_function=GeminiEmbeddingFunction()
)

# Add documents (assuming 'docs' is still defined in memory from earlier)
col.add(
    ids=[f"doc{i}" for i in range(len(docs))],
    documents=docs,
)

print(f"Stored {col.count()} documents.")

Stored 6 documents.


In [ ]:
import time

def answer(question):
    # --- PHASE 1: RETRIEVE ---
    hits = col.query(query_texts=[question], n_results=2)
    retrieved_docs = hits["documents"][0]
    distances = hits["distances"][0]
    top_distance = distances[0]

    # --- PHASE 2: GUARDRAIL ---
    if top_distance > 0.75:
        return "I don't know — that information isn't in my documents.", None

    # --- PHASE 3: AUGMENT ---
    context_parts = [f"[{i+1}] {doc}" for i, doc in enumerate(retrieved_docs)]
    context = "\n\n".join(context_parts)

    prompt = f"""Answer the user's question using ONLY the context below.
If the context does not contain the answer, you must say "I don't know."
Cite your sources using the bracketed numbers [1], [2], etc.

Context:
{context}

Question: {question}
"""

    # --- PHASE 4: GENERATE (Modern SDK + Manual Rotation) ---
    last_error = None

    # Try each key until one works
    for key in ALL_KEYS:
        try:
            # Create a temporary client for this specific key
            temp_client = genai.Client(api_key=key)

            response = temp_client.models.generate_content(
                model="gemini-3.6-flash",
                config={"thinking_config": {"thinking_level": "low"}},
                contents=prompt
            )

            # If successful, return immediately
            return response.text, context

        except Exception as e:
            # If it's a quota error (429) or invalid key (400), try the next key
            if "429" in str(e) or "400" in str(e) or "API_KEY_INVALID" in str(e):
                print(f"  [Key failed, rotating...]")
                last_error = e
                time.sleep(1) # Brief pause before trying the next key
            else:
                # If it's a different error (like a safety block), raise it immediately
                raise e

    # If all keys failed
    raise Exception(f"All API keys failed. Last error: {last_error}")

In [ ]:
questions = [
    "What's your refund policy?",        # In the docs
    "Do you ship to Australia?",         # Not in the docs (we only ship to US/CA/EU)
    "What's your favorite color?"        # Completely irrelevant (should trigger guardrail)
]

for q in questions:
    print(f"\n{'='*50}")
    print(f"Q: {q}")
    ans, ctx = answer(q)
    print(f"A: {ans}")


Q: What's your refund policy?
A: Based on the provided context, the refund policy allows returns within 30 days of purchase [1]. However, digital products cannot be refunded once they have been downloaded [2].

Q: Do you ship to Australia?
A: No, shipping is only available to the US, Canada, and the EU [1].

Q: What's your favorite color?
A: I don't know — that information isn't in my documents.


In [ ]:
import json

RUBRIC = """You are an expert evaluator scoring a customer support AI.
You will be given a Question, the Context retrieved from the database, and the AI's Answer.
Score the answer on two metrics from 1 to 5:

1. Faithfulness: Does every claim in the answer come directly from the Context?
   (1 = hallucinated everything, 5 = perfectly grounded)

2. Helpfulness: Does the answer actually address the user's question?
   (1 = completely irrelevant, 5 = perfectly answers the question)

Return ONLY a valid JSON object with this exact schema:
{
  "faithfulness": <int>,
  "helpfulness": <int>,
  "reason": "<short sentence explaining the score>"
}
"""

def judge_answer(question, context, answer):
    prompt = f"""Question: {question}

Context:
{context}

Answer given:
{answer}

Score this."""

    # We use the first key, temp=0 (or thinking=low) for consistent scoring
    response = embed_client.models.generate_content(
        model="gemini-3.6-flash",
        config={
            "system_instruction": RUBRIC,
            "thinking_config": {"thinking_level": "low"},
            "response_mime_type": "application/json",
        },
        contents=prompt,
    )

    try:
        return json.loads(response.text)
    except Exception:
        return {"faithfulness": 0, "helpfulness": 0, "reason": "Parse error"}

In [ ]:
# Good Answer
print(judge_answer(
    question="What's the refund policy?",
    context="Refunds allowed within 30 days.",
    answer="You can return items within 30 days."
))
# Expected: high faithfulness, high helpfulness

# Bad Answer (Hallucination)
print(judge_answer(
    question="What's the refund policy?",
    context="Refunds allowed within 30 days.",
    answer="You can return items within 60 days and get a 20% bonus."
))
# Expected: low faithfulness (it invented 60 days and a bonus)

{'faithfulness': 5, 'helpfulness': 5, 'reason': 'The answer accurately reflects the 30-day refund window stated in the context and directly answers the question.'}
{'faithfulness': 1, 'helpfulness': 1, 'reason': 'The answer completely contradicts the context regarding the timeframe (60 days vs 30 days) and hallucinates a 20% bonus.'}


Perfect. That’s the reality of working with live AI APIs. Sometimes it’s the code, and sometimes the server is just busy.

You now have a fully working LLM as a Judge system.

Let's look at why this is such a huge milestone:

You have a RAG pipeline (Day 21).

You have a Golden Set (Day 22).

You have an automated grader that scores your system on Faithfulness and Helpfulness without you having to read a single answer (Day 23).

If you change your chunk size tomorrow and your Faithfulness score drops from 4.5 to 2.0, you'll know instantly that you broke something.